# Home Credit Default Risk — EDA + LightGBM Model
Analysis of loan applications (`application_train.csv`, `application_test.csv`) to predict default (`TARGET = 1`) and export Power BI-ready tables.

**Steps:** 1) Load  2) Clean + feature engineering  3) EDA  4) Model (LightGBM, stratified CV)  5) Evaluation & risk bands  6) Test predictions  7) Power BI export

In [ ]:
# !pip install lightgbm scikit-learn pandas matplotlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
pd.set_option('display.max_columns', 50)
DATA_DIR = '.'   # folder containing the CSV files (e.g. '/kaggle/input/home-credit-default-risk')
OUT_DIR  = '.'

## 1) Load data

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/application_train.csv')
test  = pd.read_csv(f'{DATA_DIR}/application_test.csv')
print(train.shape, test.shape)
print(train['TARGET'].value_counts(normalize=True).round(4))
train.head()

In [ ]:
miss = train.isna().mean().sort_values(ascending=False)
print('Columns with >50% missing:', (miss > .5).sum())
miss.head(10)

## 2) Cleaning & feature engineering
`DAYS_EMPLOYED = 365243` is a placeholder (not a real value) so it is replaced with NaN.

In [ ]:
def prep(d):
    d = d.copy()
    d['DAYS_EMPLOYED'] = d['DAYS_EMPLOYED'].replace(365243, np.nan)
    d['AGE_YEARS'] = -d['DAYS_BIRTH'] / 365.25
    d['YEARS_EMPLOYED'] = -d['DAYS_EMPLOYED'] / 365.25
    d['CREDIT_INCOME_RATIO'] = d['AMT_CREDIT'] / d['AMT_INCOME_TOTAL']
    d['ANNUITY_INCOME_RATIO'] = d['AMT_ANNUITY'] / d['AMT_INCOME_TOTAL']
    d['CREDIT_TERM'] = d['AMT_ANNUITY'] / d['AMT_CREDIT']
    d['GOODS_CREDIT_RATIO'] = d['AMT_GOODS_PRICE'] / d['AMT_CREDIT']
    d['EMPLOYED_AGE_RATIO'] = d['YEARS_EMPLOYED'] / d['AGE_YEARS']
    d['EXT_MEAN'] = d[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)
    d['INCOME_PER_PERSON'] = d['AMT_INCOME_TOTAL'] / d['CNT_FAM_MEMBERS']
    return d
train, test = prep(train), prep(test)
train['AGE_GROUP'] = pd.cut(train.AGE_YEARS, [0,25,35,45,55,65,100], labels=['<25','25-34','35-44','45-54','55-64','65+'])
train['EXT_MEAN_BAND'] = pd.cut(train.EXT_MEAN, [0,.3,.4,.5,.6,.7,1], labels=['<0.3','0.3-0.4','0.4-0.5','0.5-0.6','0.6-0.7','>0.7'])
train['INCOME_BAND'] = pd.qcut(train.AMT_INCOME_TOTAL, 5, labels=['Q1 Lowest','Q2','Q3','Q4','Q5 Highest'])

## 3) Exploratory analysis — default rate by segment

In [ ]:
def rate_plot(col, ax, min_n=500, top=None, order=None):
    g = train.groupby(col, observed=True)['TARGET'].agg(['count','mean'])
    g = g[g['count'] >= min_n]
    g = g.loc[order] if order is not None else g.sort_values('mean', ascending=False)
    if top: g = g.head(top)
    (g['mean']*100).iloc[::-1].plot.barh(ax=ax, color='#2a6df4')
    ax.axvline(train.TARGET.mean()*100, color='red', ls='--', lw=1)
    ax.set_title(f'Default rate % by {col}'); ax.set_xlabel('%'); ax.set_ylabel('')

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
rate_plot('AGE_GROUP', axes[0,0], order=['<25','25-34','35-44','45-54','55-64','65+'])
rate_plot('EXT_MEAN_BAND', axes[0,1], order=['<0.3','0.3-0.4','0.4-0.5','0.5-0.6','0.6-0.7','>0.7'])
rate_plot('NAME_EDUCATION_TYPE', axes[1,0])
rate_plot('NAME_INCOME_TYPE', axes[1,1])
rate_plot('OCCUPATION_TYPE', axes[2,0], top=10)
rate_plot('REGION_RATING_CLIENT', axes[2,1])
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rate_plot('CODE_GENDER', axes[0], min_n=100)
rate_plot('NAME_CONTRACT_TYPE', axes[1])
rate_plot('INCOME_BAND', axes[2], order=['Q1 Lowest','Q2','Q3','Q4','Q5 Highest'])
plt.tight_layout(); plt.show()

In [ ]:
# Distribution of the strongest signal
train.boxplot(column='EXT_MEAN', by='TARGET', figsize=(5,4)); plt.suptitle(''); plt.title('EXT_MEAN by TARGET'); plt.show()

## 4) Modeling — LightGBM with stratified cross-validation
Out-of-fold (OOF) predictions are used for evaluation and risk bands so no row is scored by a model that saw it.

In [ ]:
feats = [c for c in train.columns if c not in ('TARGET','SK_ID_CURR','AGE_GROUP','EXT_MEAN_BAND','INCOME_BAND')]
X, Xt, y = train[feats].copy(), test[feats].copy(), train['TARGET']
for c in X.select_dtypes(['object','string']).columns:
    cats = pd.Categorical(X[c]).categories
    X[c]  = pd.Categorical(X[c],  categories=cats)
    Xt[c] = pd.Categorical(Xt[c], categories=cats)

params = dict(n_estimators=1000, learning_rate=0.05, num_leaves=31, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.4, reg_lambda=5, verbose=-1, n_jobs=-1)
N_FOLDS = 5
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=42)
oof = np.zeros(len(X)); pred = np.zeros(len(Xt)); imp = pd.Series(0.0, index=feats); aucs = []
for i, (a, b) in enumerate(skf.split(X, y)):
    m = lgb.LGBMClassifier(**params)
    m.fit(X.iloc[a], y.iloc[a], eval_set=[(X.iloc[b], y.iloc[b])], eval_metric='auc',
          callbacks=[lgb.early_stopping(50, verbose=False)])
    oof[b] = m.predict_proba(X.iloc[b])[:,1]
    pred  += m.predict_proba(Xt)[:,1] / N_FOLDS
    imp   += pd.Series(m.booster_.feature_importance('gain'), index=feats)
    aucs.append(roc_auc_score(y.iloc[b], oof[b])); print(f'Fold {i}: AUC={aucs[-1]:.4f}  trees={m.best_iteration_}')
print('OOF AUC:', round(roc_auc_score(y, oof), 4))

## 5) Evaluation

In [ ]:
fpr, tpr, _ = roc_curve(y, oof)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(fpr, tpr, label=f'LightGBM AUC={roc_auc_score(y,oof):.3f}'); ax[0].plot([0,1],[0,1],'--',c='gray')
ax[0].set_xlabel('False positive rate'); ax[0].set_ylabel('True positive rate'); ax[0].set_title('ROC (out-of-fold)'); ax[0].legend()
top = (imp/imp.sum()*100).sort_values(ascending=False).head(15)
top.iloc[::-1].plot.barh(ax=ax[1], color='#2a6df4'); ax[1].set_title('Top 15 features (gain %)')
plt.tight_layout(); plt.show()

In [ ]:
# Risk bands and deciles: does predicted risk match actual default?
train['PD_OOF'] = oof
train['RISK_BAND'] = pd.cut(train.PD_OOF, [-1,.03,.07,.15,.30,1.01],
                            labels=['1-Very Low','2-Low','3-Medium','4-High','5-Very High'])
print(train.groupby('RISK_BAND', observed=True)['TARGET'].agg(['count','mean']).round(4))
dec = pd.qcut(pd.Series(oof).rank(method='first'), 10, labels=False) + 1
dec_rate = y.groupby(dec).mean()*100
dec_rate.plot.bar(color='#e5484d', figsize=(8,3.5), title='Actual default rate % by predicted-risk decile (10 = riskiest)'); plt.show()

In [ ]:
# Threshold trade-off
rows = []
for t in [0.05, 0.08, 0.10, 0.15, 0.20]:
    p = oof >= t; tn, fp, fn, tp = confusion_matrix(y, p).ravel()
    rows.append(dict(threshold=t, recall=tp/(tp+fn), precision=tp/(tp+fp), flagged_share=p.mean()))
pd.DataFrame(rows).round(3)

## 6) Test predictions

In [ ]:
sub = pd.DataFrame({'SK_ID_CURR': test.SK_ID_CURR, 'TARGET': pred})
sub.to_csv(f'{OUT_DIR}/test_predictions.csv', index=False)
print('mean predicted PD:', round(pred.mean(), 4), '| clients with PD >= 15%:', int((pred >= .15).sum()))

## 7) Power BI export

In [ ]:
keep = ['SK_ID_CURR','TARGET','PD_OOF','RISK_BAND','NAME_CONTRACT_TYPE','CODE_GENDER','FLAG_OWN_CAR','FLAG_OWN_REALTY',
        'CNT_CHILDREN','CNT_FAM_MEMBERS','AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE','NAME_INCOME_TYPE',
        'NAME_EDUCATION_TYPE','NAME_FAMILY_STATUS','NAME_HOUSING_TYPE','OCCUPATION_TYPE','ORGANIZATION_TYPE',
        'REGION_RATING_CLIENT','AGE_YEARS','YEARS_EMPLOYED','CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','CREDIT_TERM',
        'EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','EXT_MEAN','AGE_GROUP','INCOME_BAND','EXT_MEAN_BAND']
pb = train[keep].copy()
for c in ['RISK_BAND','AGE_GROUP','INCOME_BAND','EXT_MEAN_BAND']: pb[c] = pb[c].astype(str)
pb['CODE_GENDER'] = pb.CODE_GENDER.replace({'M':'Male','F':'Female','XNA':'Unknown'})
pb.round(4).to_csv(f'{OUT_DIR}/powerbi_loans_clean.csv', index=False)
(imp/imp.sum()*100).sort_values(ascending=False).rename('Importance_Pct').round(3).rename_axis('Feature').reset_index() \
    .to_csv(f'{OUT_DIR}/powerbi_feature_importance.csv', index=False)
print('Exported Power BI files.')

## Key findings
- Default rate is ~8.1% (imbalanced) → AUC is used as the main metric.
- The mean of `EXT_SOURCE_1/2/3` is the strongest predictor; credit term (annuity/credit), organization and occupation follow.
- Highest-risk segments: age < 25, low-skill laborers, region rating 3, lower-secondary education.
- Out-of-fold AUC ≈ 0.76 with LightGBM; risk bands separate ~1.8% from ~37.6% actual default rate.
- Possible next steps: tune hyperparameters, add the other Home Credit tables (bureau, previous applications, installments), SHAP explanations, cost-based threshold.